## Notebook Overview

### Macro Dataset B Cleaning

This notebook retrieves and cleans the BFS dataset on job vacancies by economic division.

Purpose:
- fetch the dataset from the BFS PXWeb API
- convert the JSON-stat response into a pandas dataframe
- clean and standardize industry labels
- keep the most recent 8 quarters
- export a processed dataset for later integration with the micro-level job listing data

Output:
data/processed/bfs_vacancies_economic_division_clean.csv

In [ ]:
# Environment Setup

import requests
import pandas as pd
import itertools
from pathlib import Path

# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
# API Request

API_URL = "https://www.pxweb.bfs.admin.ch/api/v1/en/px-x-0602000000_103/px-x-0602000000_103.px"

query = {
    "query": [],
    "response": {"format": "json-stat2"}
}

response = requests.post(API_URL, json=query)
response.raise_for_status()

dataset_json = response.json()
print("API request successful.")

In [ ]:
# Convert JSON-stat to DataFrame

# Extract dimensions and values
dataset = dataset_json["dataset"]
value_list = dataset["value"]
dimension = dataset["dimension"]

dim_ids = dataset["id"]
dim_sizes = dataset["size"]

# Create category label lists
category_labels = []
for dim in dim_ids:
    labels = dimension[dim]["category"]["label"]
    category_labels.append(list(labels.values()))

# Build cartesian product of all dimensions
combinations = list(itertools.product(*category_labels))

# Create dataframe
df_industry = pd.DataFrame(combinations, columns=dim_ids)
df_industry["vacancies"] = value_list

print("Raw dataset shape:", df_industry.shape)
df_industry.head()

In [ ]:
# Save Raw Dataset

raw_output_path = DATA_RAW / "bfs_vacancies_economic_division_raw.csv"
df_industry.to_csv(raw_output_path, index=False)

print("Raw dataset saved to:", raw_output_path)

## Data Cleaning

The following cleaning steps are applied:

1. rename columns for readability
2. split quarter into year and quarter components
3. standardize industry labels
4. remove aggregate or unwanted categories if needed
5. sort observations consistently

In [ ]:
df_industry_clean = df_industry.copy()

# Rename columns depending on BFS dimension names
df_industry_clean = df_industry_clean.rename(columns={
    "WIR200": "industry",
    "TIME_PERIOD": "quarter",
    "OBS_VALUE": "vacancies"
})

df_industry_clean.head()

In [ ]:
# Keep quarter as period
df_industry_clean["quarter"] = pd.PeriodIndex(df_industry_clean["quarter"], freq="Q")

# Extract year for easier inspection
df_industry_clean["year"] = df_industry_clean["quarter"].dt.year

In [ ]:
# Standardize Industry Labels

industry_map = {
    "Information and communication": "ICT",
    "Financial and insurance activities": "Finance and insurance",
    "Professional, scientific and technical activities": "Professional services",
    "Transport and stockage": "Transport and storage",
    "Manufactury": "Manufacturing"
}

df_industry_clean["industry"] = (
    df_industry_clean["industry"]
    .replace(industry_map)
    .str.replace("Manufactury", "Manufacturing", regex=False)
    .str.replace("Finance and assurance", "Finance and insurance", regex=False)
    .str.replace("  ", " ", regex=False)
    .str.strip()
)

In [ ]:
# Keep last 8 quarters

latest_quarters = sorted(df_industry_clean["quarter"].unique())[-8:]
df_industry_clean = df_industry_clean[df_industry_clean["quarter"].isin(latest_quarters)].copy()

In [ ]:
# Sort Final Dataset

df_industry_clean = df_industry_clean.sort_values(
    by=["quarter", "industry"]
).reset_index(drop=True)

In [ ]:
# Validation

print("Final dataset shape:", df_industry_clean.shape)
print("\nQuarter range:")
print(df_industry_clean["quarter"].min(), "to", df_industry_clean["quarter"].max())

print("\nUnique industries:")
print(sorted(df_industry_clean["industry"].unique()))

print("\nMissing values:")
print(df_industry_clean.isna().sum())

In [ ]:
# Export Clean Dataset

final_output_path = DATA_PROCESSED / "bfs_vacancies_economic_division_clean.csv"
df_industry_clean.to_csv(final_output_path, index=False)

print("Clean dataset saved to:", final_output_path)